In [ ]:
from groq import Groq
import json
api = "XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"
model = "openai/gpt-oss-20b"

groq = Groq(api_key = api)


In [37]:
def read_file(file_name : str)->str:
    """
    Read File Content .

    arguments :
            file_name : path file for read it. 
    """
    try :
        with open(file_name , mode = 'r') as file :
            content = file.read()
            return content
    except Exception as e:
        return e

def write_in_file(content:str):
    """
    Write in files .
    arguments :
            content : text informations.
    """
    with open('output.txt' , mode = 'w' , encoding='utf-8') as file:
        file.write(content)


   

    


In [39]:
TOOLS = {'read_file':read_file ,
         'write_in_file':write_in_file
         }


tool_llm = [
    {
        "type":"function",
        "function":{"name":"read_file","description":"Read the contents of a file from the available file system or file storage.Use this tool when you need to inspect, analyze, or retrieve information from an existing file.",
                    "parameters":{'type':'object','properties':{'file_name':{'type':'string','description':'file name for file to read'}}}}
    },
        {
        "type":"function",
        "function":{"name":"write_in_file","description":"write content in file if exist and if not exsist create one and save it.",
                    "parameters":{'type':'object','properties':{'content':{'type':'string','description':'content to save in file'}}}}
    }
]

In [40]:
import json

message = [
    {
        "role": "user",
        "content": 'read this file "ai.txt" and summarize content in some points and after that write points in text file'
    }
]

while True:

    response = groq.chat.completions.create(
        model=model,
        messages=message,
        tools=tool_llm
    )

    assistant_message = response.choices[0].message

    # --------------------------------
    # No tool call -> final answer
    # --------------------------------
    if not assistant_message.tool_calls:

        print("FINAL ANSWER:")
        print(assistant_message.content)

        break

    # --------------------------------
    # Add ONLY valid assistant fields
    # --------------------------------
    assistant_msg = {
        "role": "assistant",
        "content": assistant_message.content,
        "tool_calls": []
    }

    for tool_call in assistant_message.tool_calls:

        assistant_msg["tool_calls"].append({
            "id": tool_call.id,
            "type": "function",
            "function": {
                "name": tool_call.function.name,
                "arguments": tool_call.function.arguments
            }
        })

    message.append(assistant_msg)

    # --------------------------------
    # Execute ALL requested tools
    # --------------------------------
    for tool_call in assistant_message.tool_calls:

        tool_name = tool_call.function.name

        arguments = json.loads(
            tool_call.function.arguments
        )

        print("CALLING TOOL:", tool_name)
        print("ARGUMENTS:", arguments)

        # Get Python function
        tool_function = TOOLS[tool_name]

        # Execute
        result = tool_function(**arguments)

        print("RESULT:", result)

        # --------------------------------
        # Send result back to LLM
        # --------------------------------
        message.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": str(result)
        })

CALLING TOOL: read_file
ARGUMENTS: {'file_name': 'ai.txt'}
RESULT: Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]
High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g., text, images, audio, and videos).
The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach th